# Course Recommendation Engine

This notebook demonstrates the machine learning model used to recommend courses based on a user's missing skills, categorized by difficulty. We will use **TF-IDF (Term Frequency-Inverse Document Frequency)** and **Cosine Similarity** to compute semantic closeness between missing skills and course descriptions.

In [ ]:
import pandas as pd
import os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## 1. Load the Dataset
Load our synthetic learning resources database (`courses.csv`), which specifies course names, skills covered, and difficulty levels.

In [ ]:
# Determine the path to the data folder from within this notebook
data_path = os.path.join('..', 'data', 'courses.csv')
df = pd.read_csv(data_path)

df.head()

## 2. Define the Target Inputs
Let's simulate a user trying to learn **React** and **Node.js** as their missing skills, with a target difficulty of **Medium**.

In [ ]:
missing_skills = ['react', 'node.js', 'express.js']
target_difficulty = 'Medium'

# Preprocess query into a single string
query = " ".join([s.lower() for s in missing_skills])
print("Query string:", query)

missing_skills = ['react', 'node.js', 'express.js']
target_difficulty = 'Medium'

# Preprocess query into a single string
query = " ".join([s.lower() for s in missing_skills])
print("Query string:", query)

In [ ]:
filtered_df = df[df['difficulty'].str.lower() == target_difficulty.lower()].copy()
print(f"Found {len(filtered_df)} courses with difficulty '{target_difficulty}'.")
filtered_df.head()

## 4. Feature Extraction & Similarity Computation
We use TF-IDF to create numerical embeddings of the course skills and our missing skills query. Then, we find the Top 3 matches using Cosine Similarity.

In [ ]:
vectorizer = TfidfVectorizer()

# The 'document corpus' consists of our query + the skills covered by the filtered courses
corpus = [query] + filtered_df['skills_covered'].tolist()

# Compute the TF-IDF matrix
tfidf_matrix = vectorizer.fit_transform(corpus)

# The first element is our query vector. The rest are course vectors.
# Calculate cosine similarity between our query and all courses:
similarity_scores = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:])[0]

filtered_df['match_score'] = similarity_scores

# Sort to get highest similarity
recommendations = filtered_df[filtered_df['match_score'] > 0].sort_values(by='match_score', ascending=False)
recommendations

## 5. Result Formatting
Format the recommendations to return to the Frontend.

In [ ]:
top_courses = recommendations.head(3).to_dict('records')

results = []
for c in top_courses:
    results.append({
        "name": c['course_name'],
        "skills": c['skills_covered'],
        "difficulty": c['difficulty'],
        "url": c['link'],
        "matchConfidence": round(float(c.get('match_score', 0) * 100), 1)
    })

pd.DataFrame(results)